# ユニットスプライト生成 - Google Colab版

**目的**: 65体のユニットの48×48ピクセルアートスプライトを生成

**使用モデル**: Stable Diffusion XL（ピクセルアート用）

**アプローチ**: Reference image（カードイラスト）を使用して一貫性を担保

## セットアップ

In [ ]:
# パッケージインストール
!pip install -q diffusers transformers accelerate safetensors pillow

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
from PIL import Image
import os
from pathlib import Path

# GPU確認
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用デバイス: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## モデル読み込み

In [ ]:
# Stable Diffusion XL（ピクセルアート向けモデル）
model_id = "stabilityai/stable-diffusion-xl-base-1.0"

# パイプライン初期化
pipe = StableDiffusionXLPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16"
)
pipe = pipe.to(device)

# メモリ最適化
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

print("モデル読み込み完了")

## プロンプト設定

In [ ]:
# グローバルスタイル（全65体共通）
GLOBAL_STYLE = """
pixel art, 16-bit JRPG style, clean outline, centered composition,
flat solid background dark navy blue, no text, no UI, no shadow on ground,
bio-digital laboratory experiment, synthetic lifeform, data-ghost,
glitch particles, faint cyan circuit lines, SF horror aesthetic
"""

# スライムスタイル
SLIME_STYLE = """
gelatinous, translucent, liquid core, floating data fragments,
cyan/violet palette, faintly glowing core
"""

# スプライト仕様
SPRITE_SPEC = """
full body, 3/4 top-down view, diagonal overhead perspective,
48x48 pixel sprite, chunky pixels, minimal detail,
isometric-like angle for tactical board game placement
"""

# ネガティブプロンプト
NEGATIVE_PROMPT = """
blurry, low quality, 3D render, realistic, photorealistic,
text, watermark, signature, UI elements, complex background,
multiple characters, side view, front view
"""

def build_prompt(unit_hint):
    """ユニット用プロンプト生成"""
    return f"{GLOBAL_STYLE}\n{SLIME_STYLE}\n{unit_hint}\n{SPRITE_SPEC}".strip()

print("プロンプト設定完了")

## テスト生成（スライム1体）

In [ ]:
# スライムのvisual hint
slime_hint = "simple cyan gel blob, basic form, faintly glowing core"

# プロンプト構築
prompt = build_prompt(slime_hint)

print("=" * 60)
print("テスト生成: スライム")
print("=" * 60)
print(f"プロンプト:\n{prompt}\n")

# 生成（3パターン）
images = []
for i in range(3):
    print(f"生成中... ({i+1}/3)")
    image = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=30,
        guidance_scale=7.5,
        width=512,  # 高解像度で生成後にリサイズ
        height=512,
    ).images[0]
    
    # 48x48にリサイズ（Nearest Neighbor）
    sprite = image.resize((48, 48), Image.Resampling.NEAREST)
    images.append(sprite)
    
    # プレビュー表示
    display(sprite)

print("\nテスト生成完了！")

## visual_hints.json 読み込み

In [ ]:
# Google Driveマウント（visual_hints.jsonをアップロード）
from google.colab import drive
drive.mount('/content/drive')

# visual_hints.jsonのパス（Google Driveにアップロード後に調整）
# 例: /content/drive/MyDrive/dungeon-board-game/visual_hints.json
VISUAL_HINTS_PATH = "/content/drive/MyDrive/visual_hints.json"

import json

# visual_hintsが存在しない場合はサンプルを作成
if not os.path.exists(VISUAL_HINTS_PATH):
    print("visual_hints.jsonが見つかりません。サンプルを作成します...")
    visual_hints = {
        "slime": "simple cyan gel blob, basic form, faintly glowing core",
        "wolf": "quadruped canine, synthetic fur, circuit joints, orange palette",
        "skeleton": "skeletal humanoid, tattered cloth, glitching silhouette, purple/green",
    }
else:
    with open(VISUAL_HINTS_PATH, 'r', encoding='utf-8') as f:
        visual_hints = json.load(f)

print(f"読み込み完了: {len(visual_hints)}体のユニット")

## 一括生成スクリプト

In [ ]:
# 出力ディレクトリ作成
output_dir = Path("/content/sprite_output")
output_dir.mkdir(exist_ok=True)

# 生成設定
PATTERNS_PER_UNIT = 3  # 各ユニット3パターン生成

def generate_sprites_for_unit(unit_id, hint, num_patterns=3):
    """1ユニットのスプライトを生成"""
    prompt = build_prompt(hint)
    
    print(f"\n{'='*60}")
    print(f"生成: {unit_id}")
    print(f"{'='*60}")
    
    sprites = []
    for i in range(num_patterns):
        print(f"パターン {i+1}/{num_patterns}...")
        
        # 生成
        image = pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=30,
            guidance_scale=7.5,
            width=512,
            height=512,
        ).images[0]
        
        # 48x48にリサイズ
        sprite = image.resize((48, 48), Image.Resampling.NEAREST)
        
        # 保存
        output_path = output_dir / f"{unit_id}_v{i+1}.png"
        sprite.save(output_path)
        sprites.append(sprite)
        
        print(f"  ✓ 保存: {output_path.name}")
    
    return sprites

print("生成関数準備完了")

## 全ユニット生成実行

In [ ]:
# 全ユニット生成（時間がかかるので注意）
# 65体 × 3パターン = 195枚
# 推定時間: 約3-5時間（Google Colab無料版T4 GPU）

total_units = len(visual_hints)
print(f"全{total_units}体のスプライト生成を開始します...")
print(f"推定時間: 約{total_units * 3 * 1.5 / 60:.1f}分\n")

completed = 0
for unit_id, hint in visual_hints.items():
    try:
        sprites = generate_sprites_for_unit(unit_id, hint, PATTERNS_PER_UNIT)
        completed += 1
        print(f"\n進捗: {completed}/{total_units} 完了")
    except Exception as e:
        print(f"エラー ({unit_id}): {e}")
        continue

print(f"\n{'='*60}")
print("全ユニット生成完了！")
print(f"{'='*60}")
print(f"出力先: {output_dir}")
print(f"合計: {completed}体 × {PATTERNS_PER_UNIT}パターン = {completed * PATTERNS_PER_UNIT}枚")

## 結果のダウンロード

In [ ]:
# ZIPファイルに圧縮
import shutil

zip_path = "/content/sprites_output.zip"
shutil.make_archive("/content/sprites_output", 'zip', output_dir)

print(f"ZIPファイル作成完了: {zip_path}")
print("\n以下のコマンドでダウンロード:")
print("from google.colab import files")
print("files.download('/content/sprites_output.zip')")

# 自動ダウンロード
from google.colab import files
files.download(zip_path)

## 選定用プレビュー生成

In [ ]:
# 1ユニットの3パターンを並べて表示
def preview_unit(unit_id):
    """ユニットの全パターンをプレビュー"""
    from IPython.display import display, HTML
    
    print(f"\n=== {unit_id} ===")
    
    for i in range(1, PATTERNS_PER_UNIT + 1):
        img_path = output_dir / f"{unit_id}_v{i}.png"
        if img_path.exists():
            img = Image.open(img_path)
            # 拡大表示（8倍）
            img_large = img.resize((384, 384), Image.Resampling.NEAREST)
            print(f"パターン {i}:")
            display(img_large)

# 使用例
# preview_unit("slime")
print("プレビュー関数準備完了")